# Baybayin OCR Training (Legacy Offline Pipeline)
This notebook mirrors the original Colab workflow but is designed to run entirely offline with the local tesstrain checkout and dataset.

## Prerequisites
- Tesseract 4.1+ with the training utilities (`unicharset_extractor`, `lstmtraining`, `combine_lang_model`)
- `eng.traineddata` (or your chosen start model) stored in `tesseract_training/data/`
- Python packages already available locally: `python-bidi`, `arabic-reshaper`, `Pillow`
- Dataset extracted to `kaggle_dataset_corrected_full/`

If you are preparing a machine without network access, make sure the above requirements are installed beforehand (e.g. copy wheels / packages onto the machine).

In [1]:
import os
import platform
import shutil
import subprocess
import sys
from pathlib import Path

print(f"Python {platform.python_version()} on {platform.platform()}")
for cmd in ["tesseract", "lstmtraining", "unicharset_extractor", "combine_lang_model"]:
    path = shutil.which(cmd)
    print(f"{cmd:25s}: {path or 'NOT FOUND'}")


Python 3.10.12 on Linux-6.6.87.2-microsoft-standard-WSL2-x86_64-with-glibc2.35
tesseract                : /usr/bin/tesseract
lstmtraining             : /usr/bin/lstmtraining
unicharset_extractor     : /usr/bin/unicharset_extractor
combine_lang_model       : /usr/bin/combine_lang_model


## Configure Paths and Hyperparameters
Adjust the values below if you want to use a different dataset location, model name, or training schedule.

In [2]:
from pathlib import Path

PROJECT_ROOT = Path('..').resolve()
TESSTRAIN_DIR = PROJECT_ROOT / 'tesseract_training'
DATASET_DIR = PROJECT_ROOT / 'kaggle_dataset_corrected_full'
LANGDATA_DIR = TESSTRAIN_DIR / 'langdata'
TESSDATA_PREFIX = TESSTRAIN_DIR / 'data'

MODEL_NAME = os.environ.get('BAYBAYIN_MODEL_NAME', 'baybayin_legacy')
START_MODEL = os.environ.get('START_MODEL', 'eng')
MAX_ITERATIONS = int(os.environ.get('MAX_ITERATIONS', '15000'))

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"TESSTRAIN_DIR: {TESSTRAIN_DIR}")
print(f"DATASET_DIR:  {DATASET_DIR}")
print(f"MODEL_NAME:   {MODEL_NAME}")
print(f"START_MODEL:  {START_MODEL}")
print(f"MAX_ITERATIONS: {MAX_ITERATIONS}")

if not DATASET_DIR.exists():
    raise FileNotFoundError("Dataset directory is missing. Set DATASET_DIR to the kaggle_dataset_corrected_full folder.")
if not (TESSTRAIN_DIR / 'Makefile').exists():
    raise FileNotFoundError('Cannot find tesstrain Makefile. Check TESSTRAIN_DIR.')


PROJECT_ROOT: /home/kmbandillo/baybayin_project
TESSTRAIN_DIR: /home/kmbandillo/baybayin_project/tesseract_training
DATASET_DIR:  /home/kmbandillo/baybayin_project/kaggle_dataset_corrected_full
MODEL_NAME:   baybayin_legacy
START_MODEL:  eng
MAX_ITERATIONS: 15000


## Step 1 – Prepare the Dataset
This mirrors the CLI helper script: it sanitises ground-truth text, copies images into tesstrain’s data tree, and (optionally) adjusts DPI. The `--no-dpi` flag skips ImageMagick if it is not installed.

In [3]:
prep_cmd = [
    sys.executable,
    str(PROJECT_ROOT / 'prepare_data.py'),
    '--source', str(DATASET_DIR),
    '--base-dir', str(TESSTRAIN_DIR),
    '--model-name', MODEL_NAME,
    '--langdata-dir', str(LANGDATA_DIR),
    '--no-dpi'
]
print('Running:', ' '.join(str(x) for x in prep_cmd))
subprocess.run(prep_cmd, check=True)


Running: /usr/bin/python3 /home/kmbandillo/baybayin_project/prepare_data.py --source /home/kmbandillo/baybayin_project/kaggle_dataset_corrected_full --base-dir /home/kmbandillo/baybayin_project/tesseract_training --model-name baybayin_legacy --langdata-dir /home/kmbandillo/baybayin_project/tesseract_training/langdata --no-dpi
### Step 0: Configuration ###
 source_gt_dir=/home/kmbandillo/baybayin_project/kaggle_dataset_corrected_full
 base_dir=/home/kmbandillo/baybayin_project/tesseract_training
 model_name=baybayin_legacy
 langdata_dir=/home/kmbandillo/baybayin_project/tesseract_training/langdata

### Step 2: Sanitizing ground truth files ###
✅ Ground truth files sanitized and copied to: /home/kmbandillo/baybayin_project/kaggle_dataset_corrected_full_clean
### Step 3: Creating Baybayin wordlist ###
✅ All 177000 sanitized ground truth files are organized at /home/kmbandillo/baybayin_project/tesseract_training/data/baybayin_legacy-ground-truth.
Note: final traineddata not found; training

CompletedProcess(args=['/usr/bin/python3', '/home/kmbandillo/baybayin_project/prepare_data.py', '--source', '/home/kmbandillo/baybayin_project/kaggle_dataset_corrected_full', '--base-dir', '/home/kmbandillo/baybayin_project/tesseract_training', '--model-name', 'baybayin_legacy', '--langdata-dir', '/home/kmbandillo/baybayin_project/tesseract_training/langdata', '--no-dpi'], returncode=0)

## Step 2 – Generate `all-gt` and `unicharset`
We build an aggregated ground-truth file to avoid hitting the shell argument-length limit, then call `unicharset_extractor` locally.

In [6]:
import subprocess

gt_dir = TESSTRAIN_DIR / 'data' / f'{MODEL_NAME}-ground-truth'
output_dir = TESSTRAIN_DIR / 'data' / MODEL_NAME
output_dir.mkdir(parents=True, exist_ok=True)
aggregate_gt = output_dir / 'all-gt'

files = sorted(gt_dir.glob('*.gt.txt'))
if not files:
    raise RuntimeError(f'No ground-truth files found in {gt_dir}. Did prepare_data.py complete successfully?')

with aggregate_gt.open('w', encoding='utf-8') as out:
    for idx, path in enumerate(files, start=1):
        text = path.read_text(encoding='utf-8').rstrip('\n')
        out.write(text + '\n')

print(f'Wrote {len(files)} lines to {aggregate_gt}')

unicharset_path = output_dir / 'unicharset'
cmd = [
    'unicharset_extractor',
    '--output_unicharset', str(unicharset_path),
    '--norm_mode', '2',
    str(aggregate_gt)
]
print('Running:', ' '.join(cmd))
subprocess.run(cmd, cwd=TESSTRAIN_DIR, check=True)
print(f'Unicharset saved to {unicharset_path}')


Wrote 59000 lines to /home/kmbandillo/baybayin_project/tesseract_training/data/baybayin_legacy/all-gt
Running: unicharset_extractor --output_unicharset /home/kmbandillo/baybayin_project/tesseract_training/data/baybayin_legacy/unicharset --norm_mode 2 /home/kmbandillo/baybayin_project/tesseract_training/data/baybayin_legacy/all-gt
Unicharset saved to /home/kmbandillo/baybayin_project/tesseract_training/data/baybayin_legacy/unicharset


Bad box coordinates in boxfile string! ᜀ
Extracting unicharset from plain text file /home/kmbandillo/baybayin_project/tesseract_training/data/baybayin_legacy/all-gt
Wrote unicharset file /home/kmbandillo/baybayin_project/tesseract_training/data/baybayin_legacy/unicharset


## Step 3 – Launch Training
This invokes the tesstrain Makefile locally using the settings above.

In [7]:
env = os.environ.copy()
env['TESSDATA_PREFIX'] = str(TESSDATA_PREFIX)
training_cmd = [
    'make',
    'training',
    f'MODEL_NAME={MODEL_NAME}',
    f'START_MODEL={START_MODEL}',
    f'TESSDATA={TESSDATA_PREFIX}',
    f'MAX_ITERATIONS={MAX_ITERATIONS}'
]
print('Running:', ' '.join(str(x) for x in training_cmd))
subprocess.run(training_cmd, cwd=TESSTRAIN_DIR, env=env, check=True)


Running: make training MODEL_NAME=baybayin_legacy START_MODEL=eng TESSDATA=/home/kmbandillo/baybayin_project/tesseract_training/data MAX_ITERATIONS=15000
You are using make version: 4.3
combine_tessdata -u /home/kmbandillo/baybayin_project/tesseract_training/data/eng.traineddata data/eng/baybayin_legacy
Extracting tessdata components from /home/kmbandillo/baybayin_project/tesseract_training/data/eng.traineddata
Wrote data/eng/baybayin_legacy.lstm
Wrote data/eng/baybayin_legacy.lstm-punc-dawg
Wrote data/eng/baybayin_legacy.lstm-word-dawg
Wrote data/eng/baybayin_legacy.lstm-number-dawg
Wrote data/eng/baybayin_legacy.lstm-unicharset
Wrote data/eng/baybayin_legacy.lstm-recoder
Wrote data/eng/baybayin_legacy.version


Version string:4.00.00alpha:eng:synth20170629:[1,36,0,1Ct3,3,16Mp3,3Lfys64Lfx96Lrx96Lfx512O1c1]
17:lstm:size=11689099, offset=192
18:lstm-punc-dawg:size=4322, offset=11689291
19:lstm-word-dawg:size=3694794, offset=11693613
20:lstm-number-dawg:size=4738, offset=15388407
21:lstm-unicharset:size=6360, offset=15393145
22:lstm-recoder:size=1012, offset=15399505
23:version:size=80, offset=15400517
Bad box coordinates in boxfile string! ᜀ
Extracting unicharset from plain text file data/baybayin_legacy/all-gt


unicharset_extractor --output_unicharset "data/baybayin_legacy/my.unicharset" --norm_mode 2 "data/baybayin_legacy/all-gt"
merge_unicharsets data/eng/baybayin_legacy.lstm-unicharset data/baybayin_legacy/my.unicharset "data/baybayin_legacy/unicharset"
Loaded unicharset of size 112 from file data/eng/baybayin_legacy.lstm-unicharset
Loaded unicharset of size 23 from file data/baybayin_legacy/my.unicharset
Wrote unicharset file data/baybayin_legacy/unicharset.


Wrote unicharset file data/baybayin_legacy/my.unicharset


PYTHONIOENCODING=utf-8 python3 generate_wordstr_box.py -i "data/baybayin_legacy-ground-truth/do_du_517.tif" -t "data/baybayin_legacy-ground-truth/do_du_517.gt.txt" > "data/baybayin_legacy-ground-truth/do_du_517.box"
tesseract "data/baybayin_legacy-ground-truth/do_du_517.tif" data/baybayin_legacy-ground-truth/do_du_517 --psm 10 lstm.train
PYTHONIOENCODING=utf-8 python3 generate_wordstr_box.py -i "data/baybayin_legacy-ground-truth/yo_yu_471.tif" -t "data/baybayin_legacy-ground-truth/yo_yu_471.gt.txt" > "data/baybayin_legacy-ground-truth/yo_yu_471.box"


...

tesseract "data/baybayin_legacy-ground-truth/bo_bu_11.tif" data/baybayin_legacy-ground-truth/bo_bu_11 --psm 10 lstm.train
PYTHONIOENCODING=utf-8 python3 generate_wordstr_box.py -i "data/baybayin_legacy-ground-truth/lo_lu_586.tif" -t "data/baybayin_legacy-ground-truth/lo_lu_586.gt.txt" > "data/baybayin_legacy-ground-truth/lo_lu_586.box"


Tesseract Open Source OCR Engine v4.1.1 with Leptonica
Page 1


tesseract "data/baybayin_legacy-ground-truth/lo_lu_586.tif" data/baybayin_legacy-ground-truth/lo_lu_586 --psm 10 lstm.train


Tesseract Open Source OCR Engine v4.1.1 with Leptonica
Page 1


KeyboardInterrupt: 

## Step 4 – Inspect Results
View the last few entries of `training.log` and confirm the final `.traineddata` artefact.

In [ ]:
log_path = TESSTRAIN_DIR / 'data' / MODEL_NAME / 'training.log'
if log_path.exists():
    print(f'Last lines of {log_path}:')
    tail = list(log_path.read_text(encoding='utf-8').splitlines())[-20:]
    for line in tail:
        print(line)
else:
    print('training.log not found yet.')

model_path = TESSTRAIN_DIR / 'data' / f'{MODEL_NAME}.traineddata'
if model_path.exists():
    size_mb = model_path.stat().st_size / (1024 * 1024)
    print(f"
✅ Model ready: {model_path} ({size_mb:.2f} MB)")
else:
    print('
Model file not created yet.')


## Next Steps
- Use `make traineddata MODEL_NAME=<name>` to regenerate deployment artefacts from intermediate checkpoints.
- Run evaluation scripts in `tesseract_training/holdout_visuals/` to visualise predictions on held-out samples.
- Archive the `training.log` and `data/<MODEL_NAME>/` directory for reproducibility.